运行约定：以本 Notebook 所在目录作为工作目录，从第一单元顺序运行。首个代码单元仅建立数据与输出目录。历史输出保留，不代表当前环境已验证。


In [ ]:
from pathlib import Path
Path('../../data').mkdir(parents=True, exist_ok=True)
Path('../../outputs/pytorch').mkdir(parents=True, exist_ok=True)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# 设置随机种子
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# 一、自然语言情感分析任务为例

> **本 Notebook 为 PyTorch 版本**，使用 PyTorch + transformers 替代原 PaddlePaddle + PaddleNLP 实现。

人类自然语言具有高度的复杂性，相同的对话在不同的情景，不同的情感，不同的人演绎，表达的效果往往也会迥然不同。

人类自然语言不只具有复杂性，同时也蕴含着丰富的情感色彩：表达人的情绪（如悲伤、快乐）、表达人的心情（如倦怠、忧郁）、表达人的喜好（如喜欢、讨厌）、表达人的个性特征和表达人的立场等等。

简单的说，我们可以将情感分析（sentiment classification）任务定义为一个分类问题，即指定一个文本输入，机器通过对文本进行分析、处理、归纳和推理后自动输出结论，如**图**所示。
<center><img src="https://ai-studio-static-online.cdn.bcebos.com/a9a35e01fc7b46b58b13e921207331cc895c7befdc964a0da088acc88479cbc0" width="800" ></center>
<br><center>图：情感分析任务</center></br>

通常情况下，人们把情感分析任务看成一个三分类问题，如 **图** 所示：
<br></br>
<center><img src="https://ai-studio-static-online.cdn.bcebos.com/b630901b397e4e7a8e78ab1d306dfa1fc070d91015a64ef0b8d590aaa8cfde14" width="600" ></center>
<br><center>图：情感分析任务</center></br>

- **正向：** 表示正面积极的情感，如高兴，幸福，惊喜，期待等。
- **负向：** 表示负面消极的情感，如难过，伤心，愤怒，惊恐等。
- **其他：** 其他类型的情感。

在情感分析任务中，研究人员除了分析句子的情感类型外，还细化到以句子中具体的"方面"为分析主体进行情感分析（aspect-level），如下：

> 这个薯片口味有点咸，太辣了，不过口感很脆。

关于薯片的口味方面是一个负向评价（咸，太辣），然而对于口感方面却是一个正向评价（很脆）。

> 我很喜欢夏威夷，就是这边的海鲜太贵了。

关于夏威夷是一个正向评价（喜欢），然而对于夏威夷的海鲜却是一个负向评价（价格太贵）。

## 1.1 情感分析任务的挑战

我们已经学习了通过把每个单词转换成向量的方式，可以完成单词语义计算任务。  
然而由于自然语言变幻莫测，我们在使用神经网络处理句子的时候，往往会遇到如下两类问题：  

- **变长句子的规整化处理，：** 自然语言句子往往是变长的，不同的句子长度可能差别很大。主流的神经网络（如RNN, LSTM, Transformer）在计算时，其输入通常要求是一个规整的多维张量，长度是固定的。对于一个batch的句子来说，最自然的表示就是[batch_size, sequence_length, feature_dim]的三维张量。这就要求sequence_length是固定的。

- **词序敏感的语义建模：** 自然语言句子往往对结构非常敏感，有时稍微颠倒单词的顺序都可能改变这句话的意思，比如：

  >你等一下我做完作业就走。
  >
  >我等一下你做完工作就走。

  >我不爱吃你做的饭。
  >
  >你不爱吃我做的饭。
  
  >我瞅你咋地。
  >
  >你瞅我咋地。
  
因此，我们需要找到一个可以考虑词和词之间顺序（关系）的神经网络，用于更好地实现自然语言句子建模。

## 1.2 变长句子的规整化处理

在使用神经网络处理变长数据时，需要先设置一个全局变量max_seq_len，再对语料中的句子进行处理，将不同的句子组成mini-batch，用于神经网络学习和处理。

**1. 设置全局变量**

设定一个全局变量max_seq_len，用来控制神经网络最大可以处理文本的长度。我们可以先观察语料中句子的分布，再设置合理的max_seq_len值，以最高的性价比完成句子分类任务。

**2. 对语料中的句子进行处理**

我们通常采用 **截断+填充** 的方式，对语料中的句子进行处理，将不同长度的句子组成mini-batch，以便让句子转换成一个张量给神经网络进行计算，如 **图** 所示。
<br></br>
<center><img src="https://ai-studio-static-online.cdn.bcebos.com/b539dd187fce48b7b4083f4917d28542de241257331d44efb688ad613281f067" width="400" ></center>
<br><center>图：变长数据处理</center></br>

* 对于长度超过max_seq_len的句子，通常会把这个句子进行截断，以便可以输入到一个张量中。句子截断是有技巧的，有时截取句子的前一部分会比后一部分好，有时则恰好相反。
  - **前向截断：** "晚饭， 真， 难， 以， 下， 咽"
  - **后向截断：**"今天， 的， 晚饭， 真， 难， 以"
* 对于句子长度不足max_seq_len的句子，我们一般会使用一个特殊的词语对这个句子进行填充，这个过程称为Padding。假设给定一个句子"我，爱，人工，智能"，max_seq_len=6，那么可能得到两种填充方式：
  - **前向填充：** "[pad]，[pad]，我，爱，人工，智能"
  - **后向填充：**"我，爱，人工，智能，[pad]，[pad]"

同样，不同的填充方式也对网络训练效果有一定影响。一般来说，后向填充是更常用的选择。

## 1.3 词序敏感的语义建模

向量化中，学习了如何学习每个单词的语义信息，从上面的举例中我们也会观察到，一个句子中词的顺序往往对这个句子的整体语义有重要的影响。因此，在刻画整个句子的语义信息过程中，不能撇开顺序信息。如果简单粗暴地把这个句子中所有词的向量做加和，会使得我们的模型无法区分句子的真实含义，例如：

>我不爱吃你做的饭。
>
>你不爱吃我做的饭。

一个有趣的想法，把一个自然语言句子看成一个序列，把自然语言的生成过程看成是一个序列生成的过程。例如对于句子"我，爱，人工，智能"，这句话的生成概率$\text{P}(\text{我，爱，人工，智能})$可以被表示为：

$\text{P}(\text{我，爱，人工，智能})=\text{P}(我|\text{<s>})*\text{P}(爱|\text{<s>，我})* \text{P}(人工|\text{<s>，我，爱})* \text{P}(智能|\text{<s>，我，爱，人工})* \text{P}(\text{</s>}|\text{<s>，我，爱，人工，智能})$

其中$\text{<s>}$和$\text{</s>}$是两个特殊的不可见符号，表示一个句子在逻辑上的开始和结束。

上面的公式把一个句子的生成过程建模成一个序列的决策过程，这就是香农在1950年左右提出的使用马尔可夫过程建模自然语言的思想。使用序列的视角看待和建模自然语言有一个明显的好处，那就是在对每个词建模的过程中，都有机会去学习这个词和之前生成的词之间的关系，并利用这种关系更好地处理自然语言。如 **图** 所示，生成句子"我，爱，人工"后，"智能"在下一步生成的概率就变得很高了，因为"人工智能"经常同时出现。
<center><img src="https://ai-studio-static-online.cdn.bcebos.com/b8f6029b46fa4ecca92e369952e4404dd93c36be1faf4b7ba603ac7b35fb6e4b" width="600" ></center>
<br><center>图：自然语言生成过程示意图</center></br>

通过考虑句子内部的序列关系，我们就可以清晰地区分"我不爱吃你做的菜"和"你不爱吃我做的菜"这两句话之间的联系与不同了。

事实上，目前大多数成功的自然语言模型都建立在对句子的序列化建模上。下面让我们学习两个经典的序列化建模模型：循环神经网络（Recurrent Neural Network，RNN）和长短时记忆网络（Long Short-Term Memory，LSTM）。

**循环神经网络RNN和长短时记忆网络LSTM的设计思考**

卷积神经网络的设计具备适合视觉任务"局部视野"特点，是因为视觉信息是局部有效的。例如在一张图片的1/4区域上有一只小猫，如果将图片3/4的内容遮挡，人类依然可以判断这是一只猫。

与此类似，RNN和LSTM的设计初衷是部分场景神经网络需要有"记忆"能力才能解决的任务。在自然语言处理任务中，往往一段文字中某个词的语义可能与前一段句子的语义相关，只有记住了上下文的神经网络才能很好的处理句子的语义关系。例如：
> 我一边吃着苹果，一边玩着苹果手机。

网络只有正确的记忆两个"苹果"的上下文"吃着"和"玩着…手机"，才能正确的识别两个苹果的语义，分别是水果和手机品牌。如果网络没有记忆功能，那么两个"苹果"只能归结到更高概率出现的语义上，得到一个相同的语义输出，这显然是不合理的。

如何设计神经网络的记忆功能呢？我们先了解下RNN网络是如何实现具备记忆功能的。RNN相当于将神经网络单元进行了横向连接，处理前一部分输入的RNN单元不仅有正常的模型输出，还会输出"记忆"传递到下一个RNN单元。而处于后一部分的RNN单元，不仅仅有来自于任务数据的输入，同时会接收从前一个RNN单元传递过来的记忆输入，这样就使得整个神经网络具备了"记忆"能力。

但是RNN网络只是初步实现了"记忆"功能，在此基础上科学家们又发明了一些RNN的变体，来加强网络的记忆能力。但RNN对"记忆"能力的设计是比较粗糙的，当网络处理的序列数据过长时，累积的内部信息就会越来越复杂，直到超过网络的承载能力，通俗的说"事无巨细的记录，总有一天大脑会崩溃"。为了解决这个问题，科学家巧妙的设计了一种记忆单元，称之为"长短时记忆网络（Long Short-Term Memory，LSTM）"。在每个处理单元内部，加入了输入门、输出门和遗忘门的设计，三者有明确的任务分工：

* 输入门：控制有多少输入信号会被融合；
* 遗忘门：控制有多少过去的记忆会被遗忘；
* 输出门：控制多少处理后的信息会被输出；

三者的作用与人类的记忆方式有异曲同工之处，即：

* 与当前任务无关的信息会直接过滤掉，如非常专注的开车时，人们几乎不注意沿途的风景；
* 过去记录的事情不一定都要永远记住，如令人伤心或者不重要的事，通常会很快被淡忘；
* 根据记忆和现实观察进行决策，如开车时会结合记忆中的路线和当前看到的路标，决策转弯或不做任何动作。

了解了这些关于网络设计的本质，下面进入实现方案的细节。

# 二、使用LSTM完成情感分析任务 

借助长短时记忆网络，我们可以非常轻松地完成情感分析任务。

对于每个句子，我们首先通过截断和填充的方式，把这些句子变成固定长度的向量。然后，利用长短时记忆网络，从左到右开始阅读每个句子。在完成阅读之后，我们使用长短时记忆网络的最后一个输出记忆，作为整个句子的语义信息，并直接把这个向量作为输入，送入一个分类层进行分类，从而完成对情感分析问题的神经网络建模。
<center><img src="https://ai-studio-static-online.cdn.bcebos.com/45a12932d1be43b58e4daa1e2c3f6c34363333c764064366986b86915beb4e04" width="400" ></center>
<br><center>图：LSTM完成情感分析任务流程</center></br>

接下来让我们看看如何使用 PyTorch 实现一个基于长短时记忆网络的情感分析模型。不同深度学习模型的训练过程基本一致，流程如下：

1. **数据处理**：选择需要使用的数据，并做好必要的预处理工作。
2. **网络定义**：使用 PyTorch 定义好网络结构，包括输入层，中间层，输出层，损失函数和优化算法。
3. **模型训练**：将准备好的训练集数据送入神经网络进行学习，并观察学习的过程是否正常，如损失函数值是否在降低。
4. **模型评估**：使用验证集数据测试训练好的神经网络，看看训练效果如何。
5. **模型预测**：取出一条文本数据，放入模型后进行预测，观察预测的情感倾向值。

## 2.1 数据准备

首先，需要下载语料用于模型训练和评估效果。我们使用的是 **ChnSentiCorp** 的酒店评论数据，这个数据集是一个开源的中文数据集，由训练数据、验证集和测试数据组成。

- 训练集：9601 条
- 验证集：1201 条
- 测试集：1201 条

每个数据表示的是用户对酒店的真实评价，以及用户对酒店的情感倾向（是正向还是负向）。

In [ ]:
import urllib.request
import zipfile

# 下载 ChnSentiCorp 数据集
data_url = "https://bj.bcebos.com/paddlenlp/datasets/ChnSentiCorp.zip"
data_dir = "../../data"
zip_path = os.path.join(data_dir, "ChnSentiCorp.zip")

os.makedirs(data_dir, exist_ok=True)
if not os.path.exists(zip_path):
    print("Downloading ChnSentiCorp dataset...")
    urllib.request.urlretrieve(data_url, zip_path)
    print("Download done.")
else:
    print("Dataset already exists.")

# 解压
extract_dir = os.path.join(data_dir, "ChnSentiCorp")
if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(data_dir)
    print("Extract done.")
else:
    print("Dataset already extracted.")

In [ ]:
def read_data(split='train'):
    """读取 ChnSentiCorp 数据集"""
    data_dict = {
        'train': '../../data/ChnSentiCorp/train.tsv',
        'dev': '../../data/ChnSentiCorp/dev.tsv',
        'test': '../../data/ChnSentiCorp/test.tsv'
    }
    data = []
    with open(data_dict[split], 'r', encoding='utf-8') as f:
        head = f.readline().strip().split('\t')
        for line in f:
            parts = line.strip().split('\t')
            if split == 'train':
                label, text = parts[0], parts[1]
                data.append({'text': text, 'label': label, 'qid': ''})
            elif split == 'dev':
                qid, label, text = parts[0], parts[1], parts[2]
                data.append({'text': text, 'label': label, 'qid': qid})
            elif split == 'test':
                qid, text = parts[0], parts[1]
                data.append({'text': text, 'label': '', 'qid': qid})
    return data

train_data = read_data('train')
dev_data = read_data('dev')
test_data = read_data('test')

print(f"Train: {len(train_data)}, Dev: {len(dev_data)}, Test: {len(test_data)}")

In [ ]:
# 查看几条样本
for item in dev_data[:5]:
    print(item)

## 2.2 直接使用预训练模型（BERT）快速体验

在动手从零训练 LSTM 之前，我们先体验一下**预训练模型**的强大能力。

PyTorch 生态中，`transformers` 库提供了丰富的预训练模型。我们使用 **bert-base-chinese**（中文 BERT）来完成情感分类的推理演示。BERT 通过大规模语料预训练，已经具备了很强的中文语义理解能力，只需在下游任务上微调即可取得优异效果。

In [ ]:
# 安装 transformers（如未安装）
# !pip install transformers -q

from transformers import BertTokenizer, BertForSequenceClassification

model_name = "bert-base-chinese"  # 中文 BERT 预训练模型
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 使用 CPU 进行推理演示（如有 GPU 可改为 cuda）
device_bert = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device_bert)
bert_model.eval()

print("BERT model loaded.")

In [ ]:
# 随机选择一个验证集样本进行预测
random_index = random.randint(0, len(dev_data) - 1)
sample = dev_data[random_index]
text = sample['text']
true_label = sample['label']

label_map = {0: '负面评价', 1: '正面评价'}

# 文本编码
inputs = tokenizer(
    text,
    return_tensors='pt',
    truncation=True,
    padding=True,
    max_length=512
)
inputs = {k: v.to(device_bert) for k, v in inputs.items()}

# 模型预测
with torch.no_grad():
    outputs = bert_model(**inputs)
    logits = outputs.logits
    probs = F.softmax(logits, dim=1)

predicted_label_id = torch.argmax(probs, dim=-1).item()
predicted_label = label_map[predicted_label_id]
confidence = probs[0][predicted_label_id].item()
true_label_text = label_map[int(true_label)]
is_correct = predicted_label_id == int(true_label)
status = "✓ 正确" if is_correct else "✗ 错误"

print(f"样本索引: {random_index}")
print(f"文本内容: {text}")
print(f"真实标签: {true_label_text} ({true_label})")
print(f"预测标签: {predicted_label} ({predicted_label_id})")
print(f"置信度: {confidence:.4f}")
print(f"预测状态: {status}")

## 2.3 手动训练 LSTM 模型

接下来，我们从零开始，使用 PyTorch 手动搭建一个基于 LSTM 的情感分析模型，完整经历数据处理 → 网络定义 → 训练 → 评估的全过程。

### 2.3.1 数据处理

<center><img src="https://ai-studio-static-online.cdn.bcebos.com/a60bc93816874e26ab8a1e758206a3e998b357e0966c4c199b9cab5af684ed98" width="800px" ></center>
<br><center>数据处理流程</center></br>
    
上图显示的是数据处理的流程，首先是输入文本，根据输入的文本数据进行分词，分词完以后构造词表，然后根据词表把分词后的文本映射成id的形式，然后再组装成batch形式的数据。

1.**分词**：使用结巴分词工具，把句子切分成单词的形式。

2.**构造词表**：将单词形式的句子，统计频率，按照频率取topK个单词，作为词表，剩下的低频单词舍弃，并给每个单词编一个号。

3.**转换成id**：主要是利用构造的词汇表，对分好词的数据映射成相应的id的形式。

4.**组装batch**：组装成一个batch数据（多条数据堆叠在一起），但是每个句子有长有短，对于较短的句子，我们需要补0对齐。

#### 构造词表

读取数据后，需要构造一个词典，把每个词都转化成一个ID，以便于神经网络训练。

------
**注意：**

在代码中我们使用了一个特殊的单词"[UNK]"，用于表示词表中没有覆盖到的词。之所以使用"[UNK]"这个符号，是为了处理某一些词，在测试数据中有，但训练数据没有的现象。

------

In [ ]:
import jieba
from collections import defaultdict

def build_vocab(texts, stopwords=None, min_freq=5, num_words=None, unk_token="[UNK]", pad_token="[PAD]"):
    """构建词表"""
    if stopwords is None:
        stopwords = set()
    word_counts = defaultdict(int)
    for text in texts:
        if not text:
            continue
        for word in jieba.cut(text):
            if word in stopwords:
                continue
            word_counts[word] += 1
    
    # 过滤低频词
    wcounts = [(w, c) for w, c in word_counts.items() if c >= min_freq]
    wcounts.sort(key=lambda x: x[1], reverse=True)
    
    if num_words is not None and len(wcounts) > (num_words - 2):
        wcounts = wcounts[:(num_words - 2)]
    
    sorted_voc = [pad_token, unk_token]
    sorted_voc.extend([w for w, c in wcounts])
    word_index = {w: i for i, w in enumerate(sorted_voc)}
    return word_index

# 收集训练集和验证集的文本
texts = [item['text'] for item in train_data] + [item['text'] for item in dev_data]

# 简单的停用词示例
stopwords = set(["的", "吗", "吧", "呀", "呜", "呢", "呗"])

word2idx = build_vocab(texts, stopwords, min_freq=5, unk_token="[UNK]", pad_token="[PAD]")
idx2word = {v: k for k, v in word2idx.items()}
vocab_size = len(word2idx)

print(f"Vocab size: {vocab_size}")
print(f"Top 20 words: {list(word2idx.keys())[:20]}")

#### 分词并转换成id

在完成word2id词典假设之后，我们还需要进一步处理原始语料，把语料中的所有句子都处理成ID序列。

In [ ]:
# 先通过下面的示例认识一下 jieba 分词和 id 转换
test_text = '选择珠江花园的原因就是方便，有电动扶梯直接到达海边，周围餐馆、食廊、商场、超市、摊位一应俱全。'
segment_text = list(jieba.cut(test_text))
input_ids = [word2idx.get(w, word2idx["[UNK]"]) for w in segment_text]

print("=" * 60)
print(f"原始文本: {test_text}")
print("=" * 60)
print(f"分词结果: {segment_text}")
print(f"编码结果ID: {input_ids}")
print("=" * 60)

#### 组装成 PyTorch Dataset 和 DataLoader

接下来，我们将原始语料中的每个句子通过截断和填充，转换成一个固定长度的句子，并将所有数据整理成mini-batch，用于训练模型。

在 PyTorch 中，我们通常自定义 `Dataset` 类来处理数据读取和预处理，再通过 `DataLoader` 实现多线程批量加载。

In [ ]:
class SentimentDataset(Dataset):
    """情感分析数据集"""
    def __init__(self, data, word2idx, max_seq_len=270, is_test=False):
        self.data = data
        self.word2idx = word2idx
        self.max_seq_len = max_seq_len
        self.is_test = is_test
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        words = list(jieba.cut(item['text']))
        input_ids = [self.word2idx.get(w, self.word2idx['[UNK]']) for w in words]
        valid_length = len(input_ids)
        
        # 截断
        if len(input_ids) > self.max_seq_len:
            input_ids = input_ids[:self.max_seq_len]
            valid_length = self.max_seq_len
        
        # 填充（后向填充）
        pad_len = self.max_seq_len - len(input_ids)
        if pad_len > 0:
            input_ids += [self.word2idx['[PAD]']] * pad_len
        
        input_ids = torch.LongTensor(input_ids)
        valid_length = torch.LongTensor([valid_length])
        
        if self.is_test:
            return input_ids, valid_length
        else:
            label = torch.LongTensor([int(item['label'])])
            return input_ids, valid_length, label


# 观察数据集长度分布，设置合理的 max_seq_len
all_lengths = []
for item in train_data:
    all_lengths.append(len(list(jieba.cut(item['text']))))
print(f"Max length: {max(all_lengths)}, 95% percentile: {int(np.percentile(all_lengths, 95))}")

max_seq_len = 270  # 与 Paddle 版本保持一致

# 创建 Dataset
train_dataset = SentimentDataset(train_data, word2idx, max_seq_len=max_seq_len)
dev_dataset = SentimentDataset(dev_data, word2idx, max_seq_len=max_seq_len)

# 创建 DataLoader
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# 打印一个 batch 查看数据格式
for batch in train_loader:
    input_ids, valid_lengths, labels = batch
    print(f"Input IDs shape: {input_ids.shape}")      # [batch, max_seq_len]
    print(f"Valid lengths shape: {valid_lengths.shape}")  # [batch, 1]
    print(f"Labels shape: {labels.shape}")              # [batch, 1]
    print(f"Labels: {labels.squeeze()[:20]}")
    break

### 2.3.2 网络定义

在 PyTorch 中，我们通过继承 `nn.Module` 来自定义网络结构。下面使用 `torch.nn.LSTM` API 实现一个基于 LSTM 的情感分析模型。

句子情感分析的关键技术是如何将文本表示成一个**携带语义的文本向量**。这里我们使用 **双向 LSTM（Bi-LSTM）** 对句子进行编码，取最后一层前向和后向的隐藏状态拼接后，送入全连接层进行分类。

In [ ]:
class LSTMModel(nn.Module):
    """基于双向LSTM的情感分析模型"""
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128, num_classes=2, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=word2idx['[PAD]'])
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        # 双向LSTM，hidden_size * 2
        self.fc = nn.Linear(hidden_size * 2, num_classes)
        
    def forward(self, input_ids):
        """前向传播
        Args:
            input_ids: [batch_size, seq_len]
        Returns:
            logits: [batch_size, num_classes]
        """
        # 1. Embedding: [batch, seq_len] -> [batch, seq_len, embed_dim]
        emb = self.embedding(input_ids)
        
        # 2. LSTM: [batch, seq_len, embed_dim] -> output, (hidden, cell)
        # hidden: [num_layers * 2, batch, hidden_size]
        lstm_out, (hidden, cell) = self.lstm(emb)
        
        # 3. 取最后一层的前向和反向 hidden state 拼接
        # hidden 形状: [num_layers * num_directions, batch, hidden_size]
        # 重新 reshape 便于索引
        hidden = hidden.view(self.lstm.num_layers, 2, hidden.size(1), hidden.size(2))
        last_forward = hidden[-1, 0, :, :]   # [batch, hidden_size]
        last_backward = hidden[-1, 1, :, :]  # [batch, hidden_size]
        concat = torch.cat([last_forward, last_backward], dim=1)  # [batch, hidden_size * 2]
        
        # 4. Dropout + 全连接分类
        out = self.dropout(concat)
        logits = self.fc(out)  # [batch, num_classes]
        return logits


# 初始化模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMModel(
    vocab_size=vocab_size,
    embed_dim=128,
    hidden_size=128,
    num_classes=2,
    num_layers=1,
    dropout=0.3
).to(device)

print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

### 2.3.3 模型训练

定义损失函数、优化器，并编写训练循环。在 PyTorch 中，训练流程通常遵循以下模式：

1. 将模型设为训练模式：`model.train()`
2. 清空梯度：`optimizer.zero_grad()`
3. 前向传播得到预测：`logits = model(input_ids)`
4. 计算损失：`loss = criterion(logits, labels)`
5. 反向传播：`loss.backward()`
6. 更新参数：`optimizer.step()`

In [ ]:
# 超参数设置
epochs = 5
learning_rate = 1e-3

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# 训练循环
for epoch in range(epochs):
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    for batch_idx, (input_ids, valid_lengths, labels) in enumerate(train_loader):
        input_ids = input_ids.to(device)
        labels = labels.squeeze().to(device)  # [batch]
        
        # 前向 + 反向 + 优化
        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        # 统计
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)
        
        if (batch_idx + 1) % 30 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] Batch [{batch_idx+1}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f}")
    
    train_acc = total_correct / total_samples
    avg_loss = total_loss / len(train_loader)
    print(f"\n>>> Epoch [{epoch+1}/{epochs}] Done | Avg Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f}\n")

### 2.3.4 模型评估

训练完成后，在验证集上评估模型效果。评估时需要将模型设为评估模式：`model.eval()`，并关闭梯度计算：`torch.no_grad()`。

In [ ]:
def evaluate(model, data_loader, device):
    """评估模型，返回准确率和平均损失"""
    model.eval()
    total_correct = 0
    total_samples = 0
    total_loss = 0
    
    with torch.no_grad():
        for input_ids, valid_lengths, labels in data_loader:
            input_ids = input_ids.to(device)
            labels = labels.squeeze().to(device)
            
            logits = model(input_ids)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
    
    avg_loss = total_loss / len(data_loader)
    accuracy = total_correct / total_samples
    return avg_loss, accuracy

val_loss, val_acc = evaluate(model, dev_loader, device)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

### 2.3.5 模型预测

取出一条文本数据，放入模型后进行预测，观察预测的情感倾向值。

In [ ]:
# 随机选择一条验证集样本进行预测
random_index = random.randint(0, len(dev_data) - 1)
sample = dev_data[random_index]
text = sample['text']
true_label = sample['label']

# 预处理
words = list(jieba.cut(text))
input_ids = [word2idx.get(w, word2idx['[UNK]']) for w in words]
valid_length = len(input_ids)
if len(input_ids) < max_seq_len:
    input_ids += [word2idx['[PAD]']] * (max_seq_len - len(input_ids))
else:
    input_ids = input_ids[:max_seq_len]
    valid_length = max_seq_len

input_tensor = torch.LongTensor([input_ids]).to(device)

# 预测
model.eval()
with torch.no_grad():
    logits = model(input_tensor)
    probs = F.softmax(logits, dim=1)

predicted_label_id = torch.argmax(probs, dim=1).item()
confidence = probs[0][predicted_label_id].item()

label_map = {0: '负面评价', 1: '正面评价'}
predicted_label = label_map[predicted_label_id]
true_label_text = label_map[int(true_label)]
is_correct = predicted_label_id == int(true_label)
status = "✓ 正确" if is_correct else "✗ 错误"

print(f"样本索引: {random_index}")
print(f"文本内容: {text}")
print(f"真实标签: {true_label_text} ({true_label})")
print(f"预测标签: {predicted_label} ({predicted_label_id})")
print(f"置信度: {confidence:.4f}")
print(f"预测状态: {status}")

## 三、总结

在本节中，我们使用 PyTorch 完成了完整的中文情感分析任务，主要内容包括：

1. **数据准备**：下载并读取 ChnSentiCorp 酒店评论数据集。
2. **预训练模型体验**：使用 `transformers` 库的 **bert-base-chinese** 快速完成情感分类推理，感受预训练模型的强大能力。
3. **手动训练 LSTM**：
   - **数据处理**：结巴分词 → 构建词表 → 文本转 ID → Padding → PyTorch Dataset/DataLoader。
   - **网络定义**：使用 `nn.Embedding` + 双向 `nn.LSTM` + `nn.Linear` 构建分类模型。
   - **模型训练**：标准的 PyTorch 训练循环（zero_grad → forward → loss → backward → step）。
   - **模型评估**：在验证集上计算准确率和损失。
   - **单条预测**：对任意输入文本进行情感倾向预测。

通过本项目的学习，你应该已经掌握了：
- 中文 NLP 的完整数据预处理流程
- LSTM 网络结构及其在文本分类中的应用
- PyTorch 中 Dataset、DataLoader、nn.Module 的标准使用方式
- 预训练语言模型（如 BERT）在下游任务中的使用方式

这些知识和技能是进入更复杂的 NLP 任务（如文本生成、阅读理解、对话系统）的重要基础。